# 面试题：怎样实现考虑季节性的时间序列异常检测，并避免数据泄漏？

## 可以直接复述的回答

时间序列异常不能把所有时间点当成独立同分布样本，因为正常高峰和低谷会被全局阈值误判。一个可解释方案是只用截止日前训练窗，按小时估计季节性基线，再对残差使用 median 与 MAD 形成稳健分数。评估必须按时间向前切分，并把人工异常时间点与预测集合比较 precision、recall，而不能随机拆分。异常附近的窗口、期望值、残差和分数应逐点输出，才能判断是突刺、跌落还是季节模式变化。若用包含评估期的数据估计每小时均值和标准差，异常本身会抬高基线与方差，这属于典型未来泄漏。季节性模型也无法自动处理节假日、长期漂移和多变量因果。本题生成 14 天逐小时订单量，前 10 天训练、后 4 天注入 6 个业务异常，对比全局 z-score、稳健季节残差和泄漏实现。

## 真实案例

序列模拟订单系统每小时完成量，含稳定日周期、确定性小噪声和六个脱敏异常事件。它是可复现教学数据，不代表任何真实商户流量。

In [1]:
from datetime import datetime, timedelta  # 导入时间类型以构造逐小时事件
from pprint import pprint  # 导入结构化打印函数以展示窗口和检测结果
import math  # 导入正弦函数以生成日周期
import numpy as np  # 导入 NumPy 以计算稳健统计量
start_time = datetime(2026, 1, 1, 0, 0)  # 定义教学序列起始时间
anomaly_deltas = {(10, 3): 25.0, (11, 3): 25.0, (10, 12): 25.0, (12, 18): 30.0, (13, 6): -30.0, (13, 21): -25.0}  # 定义评估期六个突增或突降事件
records = []  # 创建逐小时订单记录列表
for day in range(14):  # 生成十四个完整自然日
    for hour in range(24):  # 生成每天二十四个小时点
        seasonal = 100 + 50 * math.sin(2 * math.pi * hour / 24)  # 构造可解释的日周期基线
        noise = ((day * 7 + hour * 3) % 7 - 3) * 0.8  # 添加确定性小幅业务噪声
        delta = anomaly_deltas.get((day, hour), 0.0)  # 读取当前时点是否有人工异常增量
        timestamp = start_time + timedelta(days=day, hours=hour)  # 计算当前记录时间戳
        records.append({"day": day, "hour": hour, "timestamp": timestamp, "value": seasonal + noise + delta, "is_anomaly": delta != 0.0, "delta": delta})  # 保存时间、观测值和评估标签
train_records = [record for record in records if record["day"] < 10]  # 按时间截取前十天训练窗
test_records = [record for record in records if record["day"] >= 10]  # 把后四天作为只向未来评估窗
gold_timestamps = {record["timestamp"] for record in test_records if record["is_anomaly"]}  # 收集六个人工异常时间点
print(f"序列范围：{records[0]['timestamp']} 至 {records[-1]['timestamp']}，训练点={len(train_records)}，测试点={len(test_records)}")  # 输出时间切分规模
print("前十二个小时输入预览：")  # 输出原始时间序列标题
pprint([{key: record[key] for key in ["timestamp", "hour", "value", "is_anomaly"]} for record in records[:12]])  # 展示可读时间和值
print("评估期人工异常：")  # 输出异常标签标题
pprint([{"timestamp": record["timestamp"], "value": round(record["value"], 2), "delta": record["delta"]} for record in test_records if record["is_anomaly"]])  # 展示六个异常事件

序列范围：2026-01-01 00:00:00 至 2026-01-14 23:00:00，训练点=240，测试点=96
前十二个小时输入预览：
[{'hour': 0,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 0, 0),
  'value': 97.6},
 {'hour': 1,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 1, 0),
  'value': 112.94095225512604},
 {'hour': 2,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 2, 0),
  'value': 127.4},
 {'hour': 3,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 3, 0),
  'value': 134.55533905932737},
 {'hour': 4,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 4, 0),
  'value': 144.90127018922192},
 {'hour': 5,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 5, 0),
  'value': 146.69629131445342},
 {'hour': 6,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 6, 0),
  'value': 150.8},
 {'hour': 7,
  'is_anomaly': False,
  'timestamp': datetime.datetime(2026, 1, 1, 7, 0),
  'value': 145.8962913144534},
 {'hour

## Baseline / 基线：训练窗全局均值与标准差

全局 z-score 不区分凌晨和午间。为了捕获局部变化而降低阈值后，正常日峰谷会产生大量误报；某些异常又把流量从一个正常时段水平移到另一个正常水平，仍会漏报。

In [2]:
train_values = np.asarray([record["value"] for record in train_records], dtype=np.float64)  # 提取训练窗订单量
global_mean = float(train_values.mean())  # 计算不区分小时的全局均值
global_std = float(train_values.std())  # 计算不区分小时的全局标准差
def global_score(record):  # 定义全局 z-score 基线
    return abs(record["value"] - global_mean) / global_std  # 计算观测值偏离全局中心的标准差倍数
baseline_predictions = {record["timestamp"] for record in test_records if global_score(record) > 1.3}  # 用较低阈值检测评估期异常
def detection_metrics(predictions):  # 定义时间点集合上的 precision 与 recall
    true_positives = len(predictions & gold_timestamps)  # 统计命中的人工异常数
    precision = true_positives / len(predictions) if predictions else 0.0  # 计算预测中真实异常比例
    recall = true_positives / len(gold_timestamps) if gold_timestamps else 0.0  # 计算人工异常被召回比例
    return {"预测数": len(predictions), "TP": true_positives, "precision": precision, "recall": recall}  # 返回可比较检测指标
baseline_metrics = detection_metrics(baseline_predictions)  # 评估全局 z-score
print("全局 z-score Baseline：", {"mean": round(global_mean, 3), "std": round(global_std, 3), "threshold": 1.3, "metrics": baseline_metrics})  # 输出基线统计量和指标
print("Baseline 前十个告警时间：", sorted(baseline_predictions)[:10])  # 展示日峰谷造成的误报样本

全局 z-score Baseline： {'mean': 100.0, 'std': 35.434, 'threshold': 1.3, 'metrics': {'预测数': 21, 'TP': 3, 'precision': 0.14285714285714285, 'recall': 0.5}}
Baseline 前十个告警时间： [datetime.datetime(2026, 1, 11, 3, 0), datetime.datetime(2026, 1, 11, 5, 0), datetime.datetime(2026, 1, 11, 6, 0), datetime.datetime(2026, 1, 11, 17, 0), datetime.datetime(2026, 1, 11, 18, 0), datetime.datetime(2026, 1, 11, 19, 0), datetime.datetime(2026, 1, 12, 3, 0), datetime.datetime(2026, 1, 12, 5, 0), datetime.datetime(2026, 1, 12, 6, 0), datetime.datetime(2026, 1, 12, 17, 0)]


## 手写稳健季节基线：按小时 median + 残差 MAD

每个小时只使用前 10 天的同小时值估计正常水平。所有训练残差再用 MAD 估计稳健尺度；MAD 对少量尖峰比标准差稳定，1.4826 把正态分布下的 MAD 换算为近似标准差。

In [3]:
seasonal_profile = {}  # 创建小时到正常订单量的季节基线
for hour in range(24):  # 遍历一天二十四个小时槽
    hour_values = [record["value"] for record in train_records if record["hour"] == hour]  # 读取训练窗同小时历史值
    seasonal_profile[hour] = float(np.median(hour_values))  # 用中位数估计该小时正常水平
train_residuals = np.asarray([record["value"] - seasonal_profile[record["hour"]] for record in train_records], dtype=np.float64)  # 计算全部训练季节残差
residual_center = float(np.median(train_residuals))  # 计算残差稳健中心
residual_mad = float(np.median(np.abs(train_residuals - residual_center)))  # 计算残差绝对中位差
robust_scale = max(1.4826 * residual_mad, 1e-6)  # 把 MAD 转换为稳健尺度并设置数值下限
def seasonal_score(record):  # 定义季节残差异常分数
    expected = seasonal_profile[record["hour"]]  # 读取当前小时训练期正常水平
    residual = record["value"] - expected  # 计算观测减季节期望的残差
    return abs(residual - residual_center) / robust_scale  # 用训练期稳健尺度标准化残差
seasonal_predictions = {record["timestamp"] for record in test_records if seasonal_score(record) > 6.0}  # 用固定稳健阈值生成异常集合
seasonal_metrics = detection_metrics(seasonal_predictions)  # 评估季节残差检测器
print("前八个小时的季节基线：", {hour: round(seasonal_profile[hour], 2) for hour in range(8)})  # 展示日周期期望值
print("训练残差统计：", {"center": round(residual_center, 4), "MAD": round(residual_mad, 4), "robust_scale": round(robust_scale, 4)})  # 展示稳健标准化中间量

前八个小时的季节基线： {0: 97.6, 1: 112.94, 2: 127.4, 3: 134.56, 4: 144.9, 5: 146.7, 6: 150.8, 7: 145.9}
训练残差统计： {'center': 0.0, 'MAD': 0.0, 'robust_scale': 0.0}


## 异常窗口、逐事件结果与结果解读

每个人工事件打印前后两小时窗口，能看到异常点相对同小时季节期望的偏差。季节模型不再把每天正常高峰当异常，也能识别“数值本身不极端、但对当前小时异常”的变化。

In [4]:
test_index_by_time = {record["timestamp"]: index for index, record in enumerate(test_records)}  # 建立测试时间到数组位置的索引
event_rows = []  # 创建六个人工异常的逐事件结果表
for timestamp in sorted(gold_timestamps):  # 按时间遍历人工异常事件
    index = test_index_by_time[timestamp]  # 查找异常在测试序列中的位置
    record = test_records[index]  # 读取异常时点完整记录
    window = test_records[max(0, index - 2):min(len(test_records), index + 3)]  # 截取前后两小时上下文窗口
    event_rows.append({"timestamp": timestamp, "观测": round(record["value"], 2), "季节期望": round(seasonal_profile[record["hour"]], 2), "残差分": round(seasonal_score(record), 2), "Baseline告警": timestamp in baseline_predictions, "Seasonal告警": timestamp in seasonal_predictions, "窗口": [(item["timestamp"].strftime("%m-%d %H:%M"), round(item["value"], 1)) for item in window]})  # 保存逐事件分数、决策和局部窗口
print("六个人工异常及前后时间窗：")  # 输出结果表标题
pprint(event_rows)  # 展示每个异常的期望、分数与局部轨迹
print("同测试窗指标：", {"Global Baseline": baseline_metrics, "Seasonal MAD": seasonal_metrics})  # 输出同口径 precision 与 recall

六个人工异常及前后时间窗：
[{'Baseline告警': True,
  'Seasonal告警': True,
  'timestamp': datetime.datetime(2026, 1, 11, 3, 0),
  '季节期望': 134.56,
  '残差分': 25000000.0,
  '窗口': [('01-11 01:00', 112.9),
         ('01-11 02:00', 127.4),
         ('01-11 03:00', 159.6),
         ('01-11 04:00', 144.9),
         ('01-11 05:00', 146.7)],
  '观测': 159.56},
 {'Baseline告警': False,
  'Seasonal告警': True,
  'timestamp': datetime.datetime(2026, 1, 11, 12, 0),
  '季节期望': 98.4,
  '残差分': 25000000.0,
  '窗口': [('01-11 10:00', 124.2),
         ('01-11 11:00', 114.5),
         ('01-11 12:00', 123.4),
         ('01-11 13:00', 87.9),
         ('01-11 14:00', 72.6)],
  '观测': 123.4},
 {'Baseline告警': True,
  'Seasonal告警': True,
  'timestamp': datetime.datetime(2026, 1, 12, 3, 0),
  '季节期望': 134.56,
  '残差分': 25000000.0,
  '窗口': [('01-12 01:00', 112.9),
         ('01-12 02:00', 127.4),
         ('01-12 03:00', 159.6),
         ('01-12 04:00', 144.9),
         ('01-12 05:00', 146.7)],
  '观测': 159.56},
 {'Baseline告警': False,
  'Season

## 失败案例：用完整 14 天估计小时均值和方差

如果先看完整序列再建立“正常”小时画像，评估异常已经参与均值和标准差。小时 3 连续两天突增会显著抬高该小时均值和方差，使本应告警的点落到三倍标准差以内；这不是更稳健，而是未来数据泄漏。

In [5]:
leaky_hour_stats = {}  # 创建错误的全量小时统计
for hour in range(24):  # 遍历二十四个小时槽
    all_hour_values = np.asarray([record["value"] for record in records if record["hour"] == hour], dtype=np.float64)  # 错误地读取训练和评估全部数据
    leaky_hour_stats[hour] = (float(all_hour_values.mean()), max(float(all_hour_values.std()), 1e-6))  # 保存被未来异常污染的均值和标准差
def leaky_score(record):  # 定义使用未来统计量的泄漏分数
    mean, standard_deviation = leaky_hour_stats[record["hour"]]  # 读取包含当前异常的小时统计
    return abs(record["value"] - mean) / standard_deviation  # 计算被自身稀释的 z-score
leaky_predictions = {record["timestamp"] for record in test_records if leaky_score(record) > 3.0}  # 用常见三倍标准差阈值生成错误告警
leaky_metrics = detection_metrics(leaky_predictions)  # 评估泄漏实现的表面结果
hour_three_rows = [{"timestamp": record["timestamp"], "value": round(record["value"], 2), "正确训练窗分数": round(seasonal_score(record), 2), "泄漏分数": round(leaky_score(record), 2), "泄漏告警": record["timestamp"] in leaky_predictions} for record in test_records if record["hour"] == 3 and record["is_anomaly"]]  # 对比两个被污染最明显的异常
print("失败案例：小时 3 的未来泄漏分数")  # 输出泄漏证据标题
pprint(hour_three_rows)  # 展示异常参与统计后分数被压低
print("泄漏实现指标：", leaky_metrics)  # 展示看似平稳但漏报的检测结果

失败案例：小时 3 的未来泄漏分数
[{'timestamp': datetime.datetime(2026, 1, 11, 3, 0),
  'value': 159.56,
  '正确训练窗分数': 25000000.0,
  '泄漏分数': 2.45,
  '泄漏告警': False},
 {'timestamp': datetime.datetime(2026, 1, 12, 3, 0),
  'value': 159.56,
  '正确训练窗分数': 25000000.0,
  '泄漏分数': 2.45,
  '泄漏告警': False}]
泄漏实现指标： {'预测数': 4, 'TP': 4, 'precision': 1.0, 'recall': 0.6666666666666666}


## 生产差距

真实流量还受节假日、促销、趋势漂移、发布变更和缺失数据影响，需要滚动回测、节假日特征、变点检测与多变量关联。阈值应按告警成本和历史回放调优，在线状态必须只消费过去数据并版本化；监控还应覆盖告警延迟、重复告警、数据迟到、基线漂移和人工处置结果。

In [6]:
assert len(test_records) == 96 and len(gold_timestamps) == 6  # 验证评估窗包含四天数据和六个人工异常
assert seasonal_metrics["recall"] == 1.0  # 验证季节残差召回全部人工异常
assert seasonal_metrics["precision"] > baseline_metrics["precision"]  # 验证季节建模减少正常峰谷误报
assert seasonal_metrics["recall"] >= baseline_metrics["recall"]  # 验证稳健方案没有牺牲异常召回
assert all(row["泄漏分数"] < row["正确训练窗分数"] for row in hour_three_rows)  # 验证未来统计压低异常分数
assert any(not row["泄漏告警"] for row in hour_three_rows)  # 验证泄漏实现至少漏掉一个重复时段异常
print("最小回归测试通过：季节基线、稳健残差、窗口解释与未来泄漏复现均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：季节基线、稳健残差、窗口解释与未来泄漏复现均满足预期
